# Does the explanation survive augmentation?

A demo: one photo under growing transformations (rotation, zoom, colour shift, noise, blur). NAVE's concepts
are fitted on the original photo only, then every transformed version is explained with them, so a concept
keeps its colour throughout. The curve below shows the share of pixels that keep their concept, once the
geometric transformations are undone.

**Needs:** `pip install "nave-xai[animation]"`. **Downloads:** none with the default ViT-Ti/16.

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai[animation] @ git+https://github.com/Ahcene-B/NAVE.git"')
# elsewhere: pip install "nave-xai[animation]"
import numpy as np

MODEL, TRAINING = "vitt16", "imagenet"    # or "resnet50", "vits16" with "dino", ...
K = 8
PHOTO = "chelsea"                          # any scikit-image photo, or your own below
FAMILIES = ("rotation", "zoom", "colour", "noise", "blur")
STEPS = 9                                  # strengths from 0 (none) to 1 (strong)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display


def match_colours(labels, reference, K):
    """Give every concept of `labels` the colour index of the reference concept it overlaps most
    (one-to-one, maximising the total overlap), so colours stay stable from frame to frame."""
    overlap = np.zeros((K, K))
    np.add.at(overlap, (labels.ravel(), reference.ravel()), 1)
    try:
        from scipy.optimize import linear_sum_assignment
        rows, cols = linear_sum_assignment(-overlap)
        mapping = dict(zip(rows, cols))
    except ImportError:                                     # greedy, without scipy
        mapping, used = {}, set()
        for a, b in sorted(np.ndindex(K, K), key=lambda ab: -overlap[ab]):
            if a not in mapping and b not in used:
                mapping[a] = b
                used.add(b)
    return np.vectorize(lambda a: mapping.get(a, a))(labels)


def knn_agreement(features, targets, k=5, n=1500, seed=0):
    """How well a point's k nearest neighbours (cosine) predict its target: leave-one-out accuracy."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(features), min(n, len(features)), replace=False)
    F = features[idx] / np.linalg.norm(features[idx], axis=1, keepdims=True)
    t = targets[idx]
    sim = F @ F.T
    np.fill_diagonal(sim, -np.inf)
    nearest = np.argsort(-sim, axis=1)[:, :k]
    votes = np.array([np.bincount(t[row], minlength=t.max() + 1).argmax() for row in nearest])
    return float((votes == t).mean())


def normalise_embedding(e):
    e = e - np.median(e, axis=0)
    return e / np.percentile(np.abs(e), 98)


def token_labels(maps, centres):
    """The concept at every patch centre of every image: one label per token."""
    return np.concatenate([m[np.ix_(centres, centres)].ravel() for m in maps])


def match_to(labels, reference, K):
    """Rename `labels` so each concept takes the id of the reference concept it overlaps most."""
    return match_colours(labels, reference, K)


In [ ]:
from skimage import data, transform, color, filters, util

def load(name, size=224):
    img = getattr(data, name)()
    img = np.repeat(img[..., None], 3, axis=2) if img.ndim == 2 else img[..., :3]
    return (transform.resize(img, (size, size), anti_aliasing=False) * 255).astype(np.uint8)

image = load(PHOTO)
# your own photo instead:
# from PIL import Image
# image = np.asarray(Image.open("my_photo.jpg").convert("RGB").resize((224, 224)))
plt.imshow(image); plt.axis("off");

## The transformations

Each family has a strength from 0 to 1. Geometric ones (rotation, zoom) also say how to move a label map
the same way, so the original explanation can be compared pixel by pixel; pixels that fall outside the
photo are left out.

In [ ]:
def augment(img, family, s):
    """The transformed photo, and a function moving a label map the same way (None: no geometry)."""
    f = img / 255.0
    if family == "rotation":
        angle = 90 * s
        out = transform.rotate(f, angle, mode="reflect")
        move = lambda lab: transform.rotate(lab.astype(float), angle, order=0, mode="constant", cval=-1,
                                            preserve_range=True).astype(int)
    elif family == "zoom":
        z = 1 + 1.5 * s                                  # up to 2.5 x, on the centre
        h = int(round(224 / z)); o = (224 - h) // 2
        crop = lambda a, order: transform.resize(a[o:o + h, o:o + h], (224, 224), order=order,
                                                 anti_aliasing=False, preserve_range=True)
        out = crop(f, 1)
        move = lambda lab: crop(lab.astype(float), 0).astype(int)
    elif family == "colour":
        hsv = color.rgb2hsv(f); hsv[..., 0] = (hsv[..., 0] + 0.5 * s) % 1.0
        out, move = color.hsv2rgb(hsv), None
    elif family == "noise":
        out, move = util.random_noise(f, var=(0.25 * s) ** 2, rng=0), None
    else:
        out, move = filters.gaussian(f, sigma=4 * s, channel_axis=-1), None
    return (np.clip(out, 0, 1) * 255).astype(np.uint8), move

## Concepts from the original, explanations of every version

In [ ]:
from nave_xai import NAVEImage

explainer = NAVEImage(MODEL, TRAINING, k=K).fit(image)          # concepts of the original photo
reference = explainer.predict(image).explanation
palette = explainer.palette / 255
strengths = np.linspace(0, 1, STEPS)
results = {}
for fam in FAMILIES:
    rows = []
    for s in strengths:
        aug, move = augment(image, fam, s)
        lab = explainer.predict(aug).explanation
        ref = reference if move is None else move(reference)
        valid = ref >= 0
        rows.append(dict(image=aug, explanation=palette[lab], agreement=float((lab[valid] == ref[valid]).mean())))
    results[fam] = rows
    print(f"{fam:9s} agreement:", " ".join(f"{r['agreement']:.2f}" for r in rows))

## The animation

One row per family: the transformed photo and its explanation; the curves give the agreement with the
original's explanation as the strength grows.

In [ ]:
def animate_augmentation(frames_per_step=3):
    n = len(FAMILIES)
    fig = plt.figure(figsize=(9, 2.4 * n), dpi=60)
    grid = fig.add_gridspec(n, 3, width_ratios=[1, 1, 1.6], hspace=0.25, wspace=0.15)
    arts = []
    for r, fam in enumerate(FAMILIES):
        a0, a1, a2 = (fig.add_subplot(grid[r, j]) for j in range(3))
        for a in (a0, a1):
            a.set_xticks([]); a.set_yticks([])
        a0.set_ylabel(fam, fontsize=10)
        if r == 0:
            a0.set_title("transformed", fontsize=9); a1.set_title("NAVE explanation", fontsize=9)
            a2.set_title("agreement with the original", fontsize=9)
        i0 = a0.imshow(results[fam][0]["image"]); i1 = a1.imshow(results[fam][0]["explanation"])
        a2.plot(strengths, [x["agreement"] for x in results[fam]], color="0.5", marker="o", ms=3)
        a2.set_ylim(0, 1.02); a2.set_xlim(0, 1); a2.tick_params(labelsize=7)
        m = a2.axvline(0, color="black", lw=1)
        arts.append((fam, i0, i1, m))
    head = fig.suptitle("", fontsize=11)

    def frame(f):
        k = min(f // frames_per_step, STEPS - 1)
        for fam, i0, i1, m in arts:
            i0.set_data(results[fam][k]["image"]); i1.set_data(results[fam][k]["explanation"])
            m.set_xdata([strengths[k]] * 2)
        head.set_text(f"strength {strengths[k]:.2f}")
        return [a for t in arts for a in t[1:]] + [head]

    return animation.FuncAnimation(fig, frame, frames=STEPS * frames_per_step, interval=150, blit=False)


plt.rcParams["animation.embed_limit"] = 100
anim = animate_augmentation()
display(HTML(anim.to_jshtml()))
anim.save("augmentation.gif", writer=animation.PillowWriter(fps=6), dpi=70)
plt.close()

## Things to look at

- which transformations the concepts resist, and which break them first;
- whether the concepts that change are the small ones, the background, or the object;
- the same with `MODEL = "resnet50"`, or a self-supervised ViT (`"vits16"`, `"dino"`).

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```